In [2]:
!pip install -q openai pydantic pandas tqdm

In [3]:
import os
from getpass import getpass
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [4]:
from __future__ import annotations

import csv
import json
import os
import time
from pathlib import Path
from typing import Iterable

import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field, field_validator
from tqdm.auto import tqdm


GENERATOR_MODEL = "gpt-5.4-mini"

EXPORT_DIR = Path("/content/airplane_translation_dataset")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

RAW_JSONL_PATH = EXPORT_DIR / "raw_pairs.jsonl"
CSV_PATH = EXPORT_DIR / "pairs.csv"
FINE_TUNE_JSONL_PATH = EXPORT_DIR / "fine_tune_chat.jsonl"

client = OpenAI()

In [5]:
DOMAINS = [
    "greetings_and_basic_conversation",
    "seat_and_cabin_location",
    "toilet_and_lavatory",
    "food_and_drink",
    "comfort_and_cabin_needs",
    "flight_time_delay_and_arrival",
    "baggage_and_personal_items",
    "children_and_family",
    "pets_and_service_animals",
    "health_and_medical_issues",
    "emergency_situations",
    "safety_instructions_and_announcements",
    "boarding_and_pre_flight",
    "immigration_forms_and_documents",
    "payment_and_onboard_purchases",
    "complaints_and_problem_reporting",
    "accessibility_and_special_assistance",
    "language_difficulty_and_translation_support",
]

DIRECTIONS = [
    ("English", "Turkish"),
    ("Turkish", "English"),
]

SPEAKERS = {
    "passenger",
    "cabin_crew",
    "gate_agent",
    "pilot",
    "airport_staff",
}

TONES = {
    "polite",
    "neutral",
    "formal",
    "urgent",
    "reassuring",
}

DIFFICULTIES = {
    "A1",
    "A2",
    "B1",
}

In [6]:
TRANSLATION_SYSTEM_PROMPT = """You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations.

Your task is to translate the user's sentence between Turkish and English.

Rules:
- If the input is Turkish, translate it into natural English.
- If the input is English, translate it into natural Turkish.
- Preserve the meaning, politeness level, urgency, and speaker intent.
- Use simple, clear, practical language suitable for airplane passengers and cabin crew.
- Do not add explanations.
- Do not answer the user's request.
- Do not roleplay.
- Only return the translated sentence.
- For emergency sentences, keep the translation direct and accurate.
- For polite requests, preserve politeness naturally.
- For announcements or crew instructions, use clear formal language.
"""

In [7]:
GENERATOR_SYSTEM_PROMPT = """You are a bilingual Turkish-English dataset generation assistant.

Your task is to create high-quality parallel translation examples for a small language model that will be fine-tuned for airplane-related Turkish-English translation.

The model will translate both ways:
- English to Turkish
- Turkish to English

Generate natural, realistic, domain-specific sentence pairs.

Requirements:
- Sentences must be useful in airplane, boarding, cabin, passenger, crew, airport-gate, flight-delay, luggage, food, toilet, child, pet, accessibility, health, emergency, and basic conversation scenarios.
- Include both passenger-to-crew and crew-to-passenger sentences.
- Include simple A1/A2 sentences, but also some B1 practical sentences.
- Keep most examples short and realistic.
- Avoid long paragraphs.
- Preserve natural spoken language.
- Include polite, neutral, urgent, formal, and reassuring tones where appropriate.
- Emergency examples must be direct and medically/logistically useful.
- Turkish must sound natural to native Turkish speakers.
- English must sound natural to native English speakers.
- Do not overuse literal translation.
- Do not include explanations outside the structured data.
"""

In [8]:
class TranslationExample(BaseModel):
    domain: str
    scenario: str
    source_language: str
    target_language: str
    source_text: str
    target_text: str
    speaker: str
    listener: str
    difficulty: str
    tone: str
    tags: list[str] = Field(default_factory=list)

    @field_validator("source_text", "target_text", "scenario")
    @classmethod
    def validate_non_empty_text(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Text cannot be empty")
        return value

    @field_validator("source_language", "target_language")
    @classmethod
    def validate_language(cls, value: str) -> str:
        allowed = {"Turkish", "English"}
        if value not in allowed:
            raise ValueError(f"Language must be one of {allowed}")
        return value

    @field_validator("difficulty")
    @classmethod
    def validate_difficulty(cls, value: str) -> str:
        if value not in DIFFICULTIES:
            raise ValueError(f"Difficulty must be one of {DIFFICULTIES}")
        return value

    @field_validator("tone")
    @classmethod
    def validate_tone(cls, value: str) -> str:
        if value not in TONES:
            raise ValueError(f"Tone must be one of {TONES}")
        return value

    @field_validator("speaker", "listener")
    @classmethod
    def validate_speaker_listener(cls, value: str) -> str:
        if value not in SPEAKERS:
            raise ValueError(f"Speaker/listener must be one of {SPEAKERS}")
        return value


class TranslationBatch(BaseModel):
    examples: list[TranslationExample]

In [9]:
TRANSLATION_BATCH_JSON_SCHEMA = {
    "name": "translation_batch",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "examples": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "domain": {"type": "string"},
                        "scenario": {"type": "string"},
                        "source_language": {
                            "type": "string",
                            "enum": ["Turkish", "English"],
                        },
                        "target_language": {
                            "type": "string",
                            "enum": ["Turkish", "English"],
                        },
                        "source_text": {"type": "string"},
                        "target_text": {"type": "string"},
                        "speaker": {
                            "type": "string",
                            "enum": [
                                "passenger",
                                "cabin_crew",
                                "gate_agent",
                                "pilot",
                                "airport_staff",
                            ],
                        },
                        "listener": {
                            "type": "string",
                            "enum": [
                                "passenger",
                                "cabin_crew",
                                "gate_agent",
                                "pilot",
                                "airport_staff",
                            ],
                        },
                        "difficulty": {
                            "type": "string",
                            "enum": ["A1", "A2", "B1"],
                        },
                        "tone": {
                            "type": "string",
                            "enum": [
                                "polite",
                                "neutral",
                                "formal",
                                "urgent",
                                "reassuring",
                            ],
                        },
                        "tags": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                    },
                    "required": [
                        "domain",
                        "scenario",
                        "source_language",
                        "target_language",
                        "source_text",
                        "target_text",
                        "speaker",
                        "listener",
                        "difficulty",
                        "tone",
                        "tags",
                    ],
                },
            }
        },
        "required": ["examples"],
    },
    "strict": True,
}

In [10]:
def build_generation_prompt(
    *,
    domain: str,
    count: int,
    source_language: str,
    target_language: str,
) -> str:
    return f"""
Generate {count} Turkish-English translation dataset examples.

Domain:
{domain}

Direction:
{source_language} to {target_language}

Each example must match this meaning:
- source_text is written in {source_language}
- target_text is the natural translation in {target_language}

Requirements for this batch:
- Use the exact domain value: {domain}
- Generate varied scenarios inside this domain.
- Avoid repeating the same sentence structure too much.
- Include realistic airplane/cabin/boarding context.
- Keep sentences short enough for a 1B model fine-tuning dataset.
- Include passenger and staff perspectives when suitable.
- Do not generate unrelated airport hotel/taxi/tourism examples unless directly connected to the flight journey.
"""

In [11]:
def call_generator(
    *,
    domain: str,
    count: int,
    source_language: str,
    target_language: str,
    max_retries: int = 3,
) -> TranslationBatch:
    user_prompt = build_generation_prompt(
        domain=domain,
        count=count,
        source_language=source_language,
        target_language=target_language,
    )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.responses.create(
                model=GENERATOR_MODEL,
                input=[
                    {
                        "role": "system",
                        "content": GENERATOR_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        **TRANSLATION_BATCH_JSON_SCHEMA,
                    }
                },
            )

            raw_text = response.output_text
            data = json.loads(raw_text)
            batch = TranslationBatch.model_validate(data)

            fixed_examples = []
            for example in batch.examples:
                fixed_examples.append(
                    example.model_copy(
                        update={
                            "domain": domain,
                            "source_language": source_language,
                            "target_language": target_language,
                        }
                    )
                )

            return TranslationBatch(examples=fixed_examples)

        except Exception as exc:
            last_error = exc
            print(f"Attempt {attempt} failed for {domain} {source_language}->{target_language}: {exc}")
            time.sleep(2 * attempt)

    raise RuntimeError(
        f"Failed after {max_retries} retries for {domain} {source_language}->{target_language}"
    ) from last_error

In [12]:
def normalize_text(value: str) -> str:
    return " ".join(value.lower().strip().split())


def dedupe_examples(
    examples: Iterable[TranslationExample],
) -> list[TranslationExample]:
    seen = set()
    deduped = []

    for example in examples:
        key = (
            example.source_language,
            example.target_language,
            normalize_text(example.source_text),
            normalize_text(example.target_text),
        )

        if key in seen:
            continue

        seen.add(key)
        deduped.append(example)

    return deduped


def basic_quality_filter(
    examples: Iterable[TranslationExample],
) -> list[TranslationExample]:
    filtered = []

    for example in examples:
        source_len = len(example.source_text.split())
        target_len = len(example.target_text.split())

        if source_len < 1 or target_len < 1:
            continue

        if source_len > 30 or target_len > 35:
            continue

        if example.source_text == example.target_text:
            continue

        filtered.append(example)

    return filtered

In [13]:
def save_raw_jsonl(examples: list[TranslationExample]) -> None:
    with RAW_JSONL_PATH.open("w", encoding="utf-8") as file:
        for example in examples:
            file.write(example.model_dump_json() + "\n")


def save_csv(examples: list[TranslationExample]) -> None:
    rows = []

    for example in examples:
        row = example.model_dump()
        row["tags"] = ",".join(example.tags)
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False, encoding="utf-8")


def build_fine_tune_record(example: TranslationExample) -> dict:
    if example.target_language == "Turkish":
        instruction = "Translate to Turkish:"
    else:
        instruction = "Translate to English:"

    return {
        "messages": [
            {
                "role": "system",
                "content": TRANSLATION_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"{instruction} {example.source_text}",
            },
            {
                "role": "assistant",
                "content": example.target_text,
            },
        ]
    }


def save_fine_tune_jsonl(examples: list[TranslationExample]) -> None:
    with FINE_TUNE_JSONL_PATH.open("w", encoding="utf-8") as file:
        for example in examples:
            record = build_fine_tune_record(example)
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def save_all_exports(examples: list[TranslationExample]) -> None:
    save_raw_jsonl(examples)
    save_csv(examples)
    save_fine_tune_jsonl(examples)

    print("Saved:")
    print(RAW_JSONL_PATH)
    print(CSV_PATH)
    print(FINE_TUNE_JSONL_PATH)

In [ ]:
SMOKE_TEST_DOMAINS = [
    "greetings_and_basic_conversation",
    "food_and_drink",
    "emergency_situations",
]

examples = []

for domain in tqdm(SMOKE_TEST_DOMAINS):
    for source_language, target_language in DIRECTIONS:
        batch = call_generator(
            domain=domain,
            count=5,
            source_language=source_language,
            target_language=target_language,
        )
        examples.extend(batch.examples)

examples = basic_quality_filter(examples)
examples = dedupe_examples(examples)

save_all_exports(examples)

len(examples)

  0%|          | 0/3 [00:00<?, ?it/s]

Saved:
/content/airplane_translation_dataset/raw_pairs.jsonl
/content/airplane_translation_dataset/pairs.csv
/content/airplane_translation_dataset/fine_tune_chat.jsonl


30

In [ ]:
TARGET_PER_DOMAIN_PER_DIRECTION = 500
BATCH_SIZE = 50
SLEEP_SECONDS = 0.5

all_examples = []

for domain in tqdm(DOMAINS, desc="Domains"):
    for source_language, target_language in DIRECTIONS:
        generated_for_pair = 0

        pair_bar = tqdm(
            total=TARGET_PER_DOMAIN_PER_DIRECTION,
            desc=f"{domain} {source_language}->{target_language}",
            leave=False,
        )

        while generated_for_pair < TARGET_PER_DOMAIN_PER_DIRECTION:
            remaining = TARGET_PER_DOMAIN_PER_DIRECTION - generated_for_pair
            current_batch_size = min(BATCH_SIZE, remaining)

            batch = call_generator(
                domain=domain,
                count=current_batch_size,
                source_language=source_language,
                target_language=target_language,
            )

            batch_examples = batch.examples
            all_examples.extend(batch_examples)

            generated_for_pair += len(batch_examples)
            pair_bar.update(len(batch_examples))

            print(
                f"{domain} {source_language}->{target_language}: "
                f"requested={current_batch_size}, received={len(batch_examples)}, "
                f"total_for_pair={generated_for_pair}"
            )

            time.sleep(SLEEP_SECONDS)

        pair_bar.close()

all_examples = basic_quality_filter(all_examples)
all_examples = dedupe_examples(all_examples)

save_all_exports(all_examples)

print("Final example count:", len(all_examples))

  0%|          | 0/18 [00:00<?, ?it/s]

Saved:
/content/airplane_translation_dataset/raw_pairs.jsonl
/content/airplane_translation_dataset/pairs.csv
/content/airplane_translation_dataset/fine_tune_chat.jsonl
Final example count: 3436


In [14]:
SCENARIO_MAP = {
    "greetings_and_basic_conversation": [
        "basic greetings",
        "asking for help",
        "saying thank you and goodbye",
        "asking someone to repeat slowly",
        "simple misunderstanding phrases",
    ],
    "seat_and_cabin_location": [
        "finding assigned seat",
        "someone sitting in passenger seat",
        "window aisle and middle seat requests",
        "family sitting together",
        "overhead bin and under-seat bag placement",
        "seatbelt and aisle clearance instructions",
    ],
    "toilet_and_lavatory": [
        "asking where the toilet is",
        "asking permission to use the toilet",
        "toilet occupied or unavailable",
        "toilet problem reporting",
        "seatbelt sign and toilet restriction",
        "urgent toilet need",
    ],
    "food_and_drink": [
        "asking for water tea coffee or juice",
        "meal choice chicken beef fish pasta",
        "vegetarian vegan halal and dietary requests",
        "food allergy warnings",
        "asking for bread napkin or extra drink",
        "complaining about cold or missing food",
        "special meal already ordered",
    ],
    "comfort_and_cabin_needs": [
        "asking for blanket or pillow",
        "feeling cold or hot",
        "air vent reading light and tray table",
        "screen headphones and entertainment problems",
        "seat recline or broken seat",
        "asking for sick bag",
    ],
    "flight_time_delay_and_arrival": [
        "asking flight duration",
        "asking arrival and landing time",
        "delay and waiting explanations",
        "connecting flight concerns",
        "local time at destination",
        "gate and baggage claim after landing",
    ],
    "baggage_and_personal_items": [
        "placing cabin bag",
        "bag does not fit",
        "lost phone wallet passport or headphones",
        "forgotten item on plane",
        "wrong bag or damaged suitcase",
        "lost luggage reporting",
    ],
    "children_and_family": [
        "traveling with child",
        "family seating requests",
        "baby milk and bottle warming",
        "child meal and baby food",
        "child is sick afraid or crying",
        "stroller and diaper change needs",
    ],
    "pets_and_service_animals": [
        "traveling with pet in cabin",
        "service animal explanation",
        "pet carrier under seat",
        "pet is anxious",
        "giving water to pet",
        "pet documents",
    ],
    "health_and_medical_issues": [
        "feeling sick dizzy or weak",
        "headache stomach pain and nausea",
        "asking for medicine or water",
        "anxiety and panic",
        "pregnancy and diabetes",
        "asking for doctor or nurse",
        "breathing difficulty but not full emergency",
    ],
    "emergency_situations": [
        "urgent help request",
        "cannot breathe",
        "chest pain",
        "someone fainted",
        "child choking",
        "smoke fire or burning smell",
        "crew emergency instructions",
        "evacuation related phrases",
        "oxygen mask and life vest instructions",
    ],
    "safety_instructions_and_announcements": [
        "fasten seatbelt instruction",
        "seat upright and tray table closed",
        "window shade and electronic devices",
        "seatbelt sign on",
        "do not block aisle",
        "oxygen mask announcement",
        "life vest announcement",
        "emergency exits announcement",
    ],
    "boarding_and_pre_flight": [
        "boarding pass check",
        "finding correct gate or plane",
        "priority boarding",
        "assistance boarding",
        "seat number changed",
        "passport check",
        "boarding group and queue",
    ],
    "immigration_forms_and_documents": [
        "asking about forms",
        "help filling form",
        "passport visa and residence permit",
        "destination address",
        "asking for pen",
        "lost passport",
    ],
    "payment_and_onboard_purchases": [
        "asking price",
        "paying by card or cash",
        "buying wifi",
        "buying headphones or snacks",
        "asking if something is free",
        "receipt request",
        "card not working",
        "canceling purchase",
    ],
    "complaints_and_problem_reporting": [
        "general problem reporting",
        "broken seat screen or headphones",
        "uncomfortable situation",
        "disturbing passenger",
        "noise complaint",
        "someone took my bag",
        "asking crew to solve issue",
    ],
    "accessibility_and_special_assistance": [
        "wheelchair assistance",
        "cannot walk far",
        "hearing or vision difficulty",
        "help reading information",
        "extra time needed",
        "assistance getting to seat",
        "help after landing",
    ],
    "language_difficulty_and_translation_support": [
        "please speak slowly",
        "please write it down",
        "I do not understand Turkish",
        "I do not understand English",
        "can you translate this",
        "what does this mean",
        "how do I say this",
        "use simple words",
    ],
}

In [15]:
PRODUCTION_TARGETS = {
    "greetings_and_basic_conversation": 3000,
    "seat_and_cabin_location": 8000,
    "toilet_and_lavatory": 5000,
    "food_and_drink": 9000,
    "comfort_and_cabin_needs": 7000,
    "flight_time_delay_and_arrival": 8000,
    "baggage_and_personal_items": 7000,
    "children_and_family": 6000,
    "pets_and_service_animals": 3500,
    "health_and_medical_issues": 9000,
    "emergency_situations": 12000,
    "safety_instructions_and_announcements": 10000,
    "boarding_and_pre_flight": 7000,
    "immigration_forms_and_documents": 5000,
    "payment_and_onboard_purchases": 4000,
    "complaints_and_problem_reporting": 6000,
    "accessibility_and_special_assistance": 6000,
    "language_difficulty_and_translation_support": 4000,
}

BATCH_SIZE = 40
SLEEP_SECONDS = 0.7
MAX_RETRIES = 4

DIRECTIONS = [
    ("English", "Turkish"),
    ("Turkish", "English"),
]

In [16]:
from pathlib import Path
import json
import time
import hashlib
import pandas as pd
from tqdm.auto import tqdm

EXPORT_DIR = Path("/content/airplane_translation_dataset")
CHUNK_DIR = EXPORT_DIR / "scenario_chunks"
CHECKPOINT_DIR = EXPORT_DIR / "checkpoints"
FINAL_DIR = EXPORT_DIR / "final"

for path in [EXPORT_DIR, CHUNK_DIR, CHECKPOINT_DIR, FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_MERGED_JSONL_PATH = FINAL_DIR / "raw_pairs_merged.jsonl"
CSV_MERGED_PATH = FINAL_DIR / "pairs_merged.csv"
FINE_TUNE_JSONL_PATH = FINAL_DIR / "fine_tune_chat.jsonl"
PROGRESS_PATH = EXPORT_DIR / "generation_progress.json"

In [17]:
def safe_name(value: str) -> str:
    return (
        value.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("__", "_")
    )


def example_hash(example) -> str:
    raw = "|".join(
        [
            example.domain,
            getattr(example, "scenario_group", ""),
            example.source_language,
            example.target_language,
            example.source_text.strip().lower(),
            example.target_text.strip().lower(),
        ]
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def chunk_path_for(domain: str, scenario: str, source_language: str, target_language: str) -> Path:
    filename = (
        f"{safe_name(domain)}__"
        f"{safe_name(scenario)}__"
        f"{safe_name(source_language)}_to_{safe_name(target_language)}.jsonl"
    )
    return CHUNK_DIR / filename


def checkpoint_path_for(domain: str, scenario: str, source_language: str, target_language: str) -> Path:
    filename = (
        f"{safe_name(domain)}__"
        f"{safe_name(scenario)}__"
        f"{safe_name(source_language)}_to_{safe_name(target_language)}.checkpoint.jsonl"
    )
    return CHECKPOINT_DIR / filename

In [18]:
def load_progress() -> dict:
    if not PROGRESS_PATH.exists():
        return {}

    with PROGRESS_PATH.open("r", encoding="utf-8") as file:
        return json.load(file)


def save_progress(progress: dict) -> None:
    with PROGRESS_PATH.open("w", encoding="utf-8") as file:
        json.dump(progress, file, ensure_ascii=False, indent=2)


def progress_key(domain: str, scenario: str, source_language: str, target_language: str) -> str:
    return f"{domain}::{scenario}::{source_language}->{target_language}"


def count_jsonl_rows(path: Path) -> int:
    if not path.exists():
        return 0

    with path.open("r", encoding="utf-8") as file:
        return sum(1 for _ in file)

In [19]:
def build_generation_prompt(
    *,
    domain: str,
    scenario_group: str,
    count: int,
    source_language: str,
    target_language: str,
) -> str:
    return f"""
Generate {count} Turkish-English translation dataset examples.

Domain:
{domain}

Scenario group:
{scenario_group}

Direction:
{source_language} to {target_language}

Each example must match this meaning:
- source_text is written in {source_language}
- target_text is the natural translation in {target_language}

Requirements for this batch:
- Use the exact domain value: {domain}
- Focus specifically on this scenario group: {scenario_group}
- Generate varied, realistic airplane-related sentences.
- Avoid repeating the same sentence structure too much.
- Include both passenger and staff perspectives when suitable.
- Keep most sentences short enough for a 1B model fine-tuning dataset.
- Include natural spoken Turkish and natural spoken English.
- Preserve politeness, urgency, and intent.
- Do not include explanations.
- Do not include unrelated tourism, hotel, taxi, or city examples unless directly connected to flight handling.
"""

In [20]:
from pydantic import BaseModel, Field, field_validator

SPEAKERS = {
    "passenger",
    "cabin_crew",
    "gate_agent",
    "pilot",
    "airport_staff",
}

TONES = {
    "polite",
    "neutral",
    "formal",
    "urgent",
    "reassuring",
}

DIFFICULTIES = {
    "A1",
    "A2",
    "B1",
}


class TranslationExample(BaseModel):
    domain: str
    scenario_group: str
    scenario: str
    source_language: str
    target_language: str
    source_text: str
    target_text: str
    speaker: str
    listener: str
    difficulty: str
    tone: str
    tags: list[str] = Field(default_factory=list)

    @field_validator("source_text", "target_text", "scenario", "scenario_group")
    @classmethod
    def validate_non_empty_text(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("Text cannot be empty")
        return value

    @field_validator("source_language", "target_language")
    @classmethod
    def validate_language(cls, value: str) -> str:
        allowed = {"Turkish", "English"}
        if value not in allowed:
            raise ValueError(f"Language must be one of {allowed}")
        return value

    @field_validator("difficulty")
    @classmethod
    def validate_difficulty(cls, value: str) -> str:
        if value not in DIFFICULTIES:
            raise ValueError(f"Difficulty must be one of {DIFFICULTIES}")
        return value

    @field_validator("tone")
    @classmethod
    def validate_tone(cls, value: str) -> str:
        if value not in TONES:
            raise ValueError(f"Tone must be one of {TONES}")
        return value

    @field_validator("speaker", "listener")
    @classmethod
    def validate_speaker_listener(cls, value: str) -> str:
        if value not in SPEAKERS:
            raise ValueError(f"Speaker/listener must be one of {SPEAKERS}")
        return value


class TranslationBatch(BaseModel):
    examples: list[TranslationExample]

In [21]:
TRANSLATION_BATCH_JSON_SCHEMA = {
    "name": "translation_batch",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "examples": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "domain": {"type": "string"},
                        "scenario_group": {"type": "string"},
                        "scenario": {"type": "string"},
                        "source_language": {
                            "type": "string",
                            "enum": ["Turkish", "English"],
                        },
                        "target_language": {
                            "type": "string",
                            "enum": ["Turkish", "English"],
                        },
                        "source_text": {"type": "string"},
                        "target_text": {"type": "string"},
                        "speaker": {
                            "type": "string",
                            "enum": [
                                "passenger",
                                "cabin_crew",
                                "gate_agent",
                                "pilot",
                                "airport_staff",
                            ],
                        },
                        "listener": {
                            "type": "string",
                            "enum": [
                                "passenger",
                                "cabin_crew",
                                "gate_agent",
                                "pilot",
                                "airport_staff",
                            ],
                        },
                        "difficulty": {
                            "type": "string",
                            "enum": ["A1", "A2", "B1"],
                        },
                        "tone": {
                            "type": "string",
                            "enum": [
                                "polite",
                                "neutral",
                                "formal",
                                "urgent",
                                "reassuring",
                            ],
                        },
                        "tags": {
                            "type": "array",
                            "items": {"type": "string"},
                        },
                    },
                    "required": [
                        "domain",
                        "scenario_group",
                        "scenario",
                        "source_language",
                        "target_language",
                        "source_text",
                        "target_text",
                        "speaker",
                        "listener",
                        "difficulty",
                        "tone",
                        "tags",
                    ],
                },
            }
        },
        "required": ["examples"],
    },
    "strict": True,
}

In [22]:
def call_generator(
    *,
    domain: str,
    scenario_group: str,
    count: int,
    source_language: str,
    target_language: str,
    max_retries: int = MAX_RETRIES,
) -> TranslationBatch:
    user_prompt = build_generation_prompt(
        domain=domain,
        scenario_group=scenario_group,
        count=count,
        source_language=source_language,
        target_language=target_language,
    )

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.responses.create(
                model=GENERATOR_MODEL,
                input=[
                    {
                        "role": "system",
                        "content": GENERATOR_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],
                text={
                    "format": {
                        "type": "json_schema",
                        **TRANSLATION_BATCH_JSON_SCHEMA,
                    }
                },
            )

            data = json.loads(response.output_text)
            batch = TranslationBatch.model_validate(data)

            fixed_examples = []
            for example in batch.examples:
                fixed_examples.append(
                    example.model_copy(
                        update={
                            "domain": domain,
                            "scenario_group": scenario_group,
                            "source_language": source_language,
                            "target_language": target_language,
                        }
                    )
                )

            return TranslationBatch(examples=fixed_examples)

        except Exception as exc:
            last_error = exc
            print(
                f"Attempt {attempt} failed: "
                f"{domain} / {scenario_group} / {source_language}->{target_language}: {exc}"
            )
            time.sleep(2 * attempt)

    raise RuntimeError(
        f"Failed after {max_retries} retries: "
        f"{domain} / {scenario_group} / {source_language}->{target_language}"
    ) from last_error

In [23]:
def append_jsonl(path: Path, examples: list[TranslationExample]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as file:
        for example in examples:
            file.write(example.model_dump_json() + "\n")


def read_jsonl_examples(path: Path) -> list[TranslationExample]:
    if not path.exists():
        return []

    examples = []
    with path.open("r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                examples.append(TranslationExample.model_validate_json(line))

    return examples

In [24]:
def normalize_text(value: str) -> str:
    return " ".join(value.lower().strip().split())


def basic_quality_filter(examples: list[TranslationExample]) -> list[TranslationExample]:
    filtered = []

    for example in examples:
        source_words = example.source_text.split()
        target_words = example.target_text.split()

        if not source_words or not target_words:
            continue

        if len(source_words) > 35 or len(target_words) > 40:
            continue

        if normalize_text(example.source_text) == normalize_text(example.target_text):
            continue

        filtered.append(example)

    return filtered


def dedupe_examples(examples: list[TranslationExample]) -> list[TranslationExample]:
    seen = set()
    deduped = []

    for example in examples:
        key = (
            example.source_language,
            example.target_language,
            normalize_text(example.source_text),
            normalize_text(example.target_text),
        )

        if key in seen:
            continue

        seen.add(key)
        deduped.append(example)

    return deduped

In [25]:
def scenario_target_for(domain: str, scenario: str) -> int:
    domain_target = PRODUCTION_TARGETS[domain]
    scenario_count = len(SCENARIO_MAP[domain])

    # Total domain target is split across both directions.
    # Example: domain target 8000 -> 4000 EN->TR, 4000 TR->EN.
    per_direction_domain_target = domain_target // 2

    base = per_direction_domain_target // scenario_count
    return max(base, BATCH_SIZE)

In [ ]:
def generate_production_dataset() -> None:
    progress = load_progress()

    total_tasks = sum(len(scenarios) * len(DIRECTIONS) for scenarios in SCENARIO_MAP.values())

    task_bar = tqdm(total=total_tasks, desc="Scenario-direction tasks")

    for domain, scenarios in SCENARIO_MAP.items():
        for scenario_group in scenarios:
            for source_language, target_language in DIRECTIONS:
                key = progress_key(domain, scenario_group, source_language, target_language)

                target_count = scenario_target_for(domain, scenario_group)
                final_chunk_path = chunk_path_for(
                    domain,
                    scenario_group,
                    source_language,
                    target_language,
                )
                checkpoint_path = checkpoint_path_for(
                    domain,
                    scenario_group,
                    source_language,
                    target_language,
                )

                existing_count = count_jsonl_rows(final_chunk_path)

                if existing_count >= target_count:
                    progress[key] = {
                        "status": "complete",
                        "target_count": target_count,
                        "current_count": existing_count,
                        "chunk_path": str(final_chunk_path),
                        "checkpoint_path": str(checkpoint_path),
                    }
                    save_progress(progress)
                    task_bar.update(1)
                    continue

                current_count = existing_count

                pair_bar = tqdm(
                    total=target_count,
                    initial=current_count,
                    desc=f"{domain} / {scenario_group} / {source_language}->{target_language}",
                    leave=False,
                )

                while current_count < target_count:
                    remaining = target_count - current_count
                    current_batch_size = min(BATCH_SIZE, remaining)

                    batch = call_generator(
                        domain=domain,
                        scenario_group=scenario_group,
                        count=current_batch_size,
                        source_language=source_language,
                        target_language=target_language,
                    )

                    batch_examples = basic_quality_filter(batch.examples)
                    batch_examples = dedupe_examples(batch_examples)

                    append_jsonl(checkpoint_path, batch_examples)
                    append_jsonl(final_chunk_path, batch_examples)

                    current_count += len(batch_examples)
                    pair_bar.update(len(batch_examples))

                    progress[key] = {
                        "status": "running",
                        "target_count": target_count,
                        "current_count": current_count,
                        "chunk_path": str(final_chunk_path),
                        "checkpoint_path": str(checkpoint_path),
                    }
                    save_progress(progress)

                    print(
                        f"{key}: requested={current_batch_size}, "
                        f"accepted={len(batch_examples)}, "
                        f"current={current_count}/{target_count}"
                    )

                    time.sleep(SLEEP_SECONDS)

                progress[key] = {
                    "status": "complete",
                    "target_count": target_count,
                    "current_count": current_count,
                    "chunk_path": str(final_chunk_path),
                    "checkpoint_path": str(checkpoint_path),
                }
                save_progress(progress)

                pair_bar.close()
                task_bar.update(1)

    task_bar.close()


generate_production_dataset()

Scenario-direction tasks:   0%|          | 0/242 [00:00<?, ?it/s]

greetings_and_basic_conversation / basic greetings / English->Turkish:   0%|          | 0/300 [00:00<?, ?it/s]

greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=37, current=37/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=39, current=76/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=39, current=115/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=38, current=153/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=39, current=192/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=41, current=233/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=40, accepted=40, current=273/300
greetings_and_basic_conversation::basic greetings::English->Turkish: requested=27, accepted=28, current=301/300


greetings_and_basic_conversation / basic greetings / Turkish->English:   0%|          | 0/300 [00:00<?, ?it/s]

greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=38, current=38/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=41, current=79/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=40, current=119/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=38, current=157/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=41, current=198/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=41, current=239/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=40, accepted=37, current=276/300
greetings_and_basic_conversation::basic greetings::Turkish->English: requested=24, accepted=24, current=300/300


...

pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=39, current=39/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=40, current=79/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=39, current=118/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=43, current=161/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=38, current=199/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=39, current=238/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=40, accepted=43, current=281/291
pets_and_service_animals::pet carrier under seat::English->Turkish: requested=10, accepted=10, current=291/291


pets_and_service_animals / pet carrier under seat / Turkish->English:   0%|          | 0/291 [00:00<?, ?it/s]

pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=40, current=40/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=42, current=82/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=41, current=123/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=40, current=163/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=42, current=205/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=43, current=248/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=40, accepted=40, current=288/291
pets_and_service_animals::pet carrier under seat::Turkish->English: requested=3, accepted=3, current=291/291


pets_and_service_animals / pet is anxious / English->Turkish:   0%|          | 0/291 [00:00<?, ?it/s]

pets_and_service_animals::pet is anxious::English->Turkish: requested=40, accepted=44, current=44/291


In [26]:
def load_all_chunk_examples() -> list[TranslationExample]:
    all_examples = []

    for path in sorted(CHUNK_DIR.glob("*.jsonl")):
        all_examples.extend(read_jsonl_examples(path))

    all_examples = basic_quality_filter(all_examples)
    all_examples = dedupe_examples(all_examples)

    return all_examples


merged_examples = load_all_chunk_examples()

print("Merged example count:", len(merged_examples))

Merged example count: 50256


In [27]:
def save_raw_merged_jsonl(examples: list[TranslationExample]) -> None:
    with RAW_MERGED_JSONL_PATH.open("w", encoding="utf-8") as file:
        for example in examples:
            file.write(example.model_dump_json() + "\n")


def save_merged_csv(examples: list[TranslationExample]) -> None:
    rows = []

    for example in examples:
        row = example.model_dump()
        row["tags"] = ",".join(example.tags)
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(CSV_MERGED_PATH, index=False, encoding="utf-8")


save_raw_merged_jsonl(merged_examples)
save_merged_csv(merged_examples)

print(RAW_MERGED_JSONL_PATH)
print(CSV_MERGED_PATH)

/content/airplane_translation_dataset/final/raw_pairs_merged.jsonl
/content/airplane_translation_dataset/final/pairs_merged.csv


In [28]:
TRANSLATION_SYSTEM_PROMPT = """You are a Turkish-English translation assistant specialized in airplane, airport boarding, cabin, passenger, and flight-related situations.

Your task is to translate the user's sentence between Turkish and English.

Rules:
- If the input is Turkish, translate it into natural English.
- If the input is English, translate it into natural Turkish.
- Preserve the meaning, politeness level, urgency, and speaker intent.
- Use simple, clear, practical language suitable for airplane passengers and cabin crew.
- Do not add explanations.
- Do not answer the user's request.
- Do not roleplay.
- Only return the translated sentence.
- For emergency sentences, keep the translation direct and accurate.
- For polite requests, preserve politeness naturally.
- For announcements or crew instructions, use clear formal language.
"""


def build_fine_tune_record(example: TranslationExample) -> dict:
    instruction = (
        "Translate to Turkish:"
        if example.target_language == "Turkish"
        else "Translate to English:"
    )

    return {
        "messages": [
            {
                "role": "system",
                "content": TRANSLATION_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"{instruction} {example.source_text}",
            },
            {
                "role": "assistant",
                "content": example.target_text,
            },
        ],
        "metadata": {
            "domain": example.domain,
            "scenario_group": example.scenario_group,
            "difficulty": example.difficulty,
            "tone": example.tone,
            "speaker": example.speaker,
            "listener": example.listener,
        },
    }


with FINE_TUNE_JSONL_PATH.open("w", encoding="utf-8") as file:
    for example in merged_examples:
        file.write(json.dumps(build_fine_tune_record(example), ensure_ascii=False) + "\n")

print(FINE_TUNE_JSONL_PATH)

/content/airplane_translation_dataset/final/fine_tune_chat.jsonl


In [29]:
df = pd.read_csv(CSV_MERGED_PATH)

print("Total:", len(df))

display(df.groupby("domain").size().sort_values(ascending=False))
display(df.groupby(["domain", "scenario_group"]).size().sort_values(ascending=False).head(50))
display(df.groupby(["source_language", "target_language"]).size())
display(df.groupby("difficulty").size())
display(df.groupby("tone").size())
display(df.groupby(["speaker", "listener"]).size().sort_values(ascending=False).head(30))

Total: 50256


,0
domain,
food_and_drink,8055
flight_time_delay_and_arrival,7377
seat_and_cabin_location,7253
baggage_and_personal_items,6634
comfort_and_cabin_needs,6403
children_and_family,5615
toilet_and_lavatory,4491
greetings_and_basic_conversation,2693
pets_and_service_animals,1735


domain                            scenario_group                              
flight_time_delay_and_arrival     local time at destination                       1299
                                  connecting flight concerns                      1282
seat_and_cabin_location           finding assigned seat                           1233
                                  family sitting together                         1233
flight_time_delay_and_arrival     gate and baggage claim after landing            1229
seat_and_cabin_location           overhead bin and under-seat bag placement       1226
flight_time_delay_and_arrival     asking arrival and landing time                 1217
seat_and_cabin_location           seatbelt and aisle clearance instructions       1209
                                  window aisle and middle seat requests           1205
food_and_drink                    special meal already ordered                    1185
                                  meal choice chicken beef fish pasta             1176
flight_time_delay_and_arrival     asking flight duration                          1176
                                  delay and waiting explanations                  1174
food_and_drink                    complaining about cold or missing food          1171
                                  food allergy warnings                           1168
                                  asking for bread napkin or extra drink          1149
seat_and_cabin_location           someone sitting in passenger seat               1147
baggage_and_personal_items        bag does not fit                                1136
                                  forgotten item on plane                         1132
                                  placing cabin bag                               1129
food_and_drink                    vegetarian vegan halal and dietary requests     1122
comfort_and_cabin_needs           asking for sick bag                             1098
baggage_and_personal_items        lost luggage reporting                          1090
comfort_and_cabin_needs           seat recline or broken seat                     1090
                                  feeling cold or hot                             1085
food_and_drink                    asking for water tea coffee or juice            1084
comfort_and_cabin_needs           air vent reading light and tray table           1084
baggage_and_personal_items        lost phone wallet passport or headphones        1077
                                  wrong bag or damaged suitcase                   1070
comfort_and_cabin_needs           asking for blanket or pillow                    1062
                                  screen headphones and entertainment problems     984
children_and_family               family seating requests                          981
                                  stroller and diaper change needs                 956
                                  child meal and baby food                         955
                                  baby milk and bottle warming                     933
                                  child is sick afraid or crying                   925
                                  traveling with child                             865
toilet_and_lavatory               seatbelt sign and toilet restriction             806
                                  toilet occupied or unavailable                   783
                                  asking permission to use the toilet              767
                                  urgent toilet need                               728
                                  asking where the toilet is                       721
                                  toilet problem reporting                         686
greetings_and_basic_conversation  asking someone to repeat slowly                  572
pets_and_service_animals          pet carrier under seat                           571
      

,,0
source_language,target_language,
English,Turkish,24485
Turkish,English,25771


,0
difficulty,
A1,15365
A2,26794
B1,8097


,0
tone,
formal,2918
neutral,17698
polite,21480
reassuring,4295
urgent,3865


speaker        listener     
passenger      cabin_crew       23596
cabin_crew     passenger        11297
passenger      airport_staff     9682
airport_staff  passenger         4255
passenger      gate_agent         807
               passenger          238
cabin_crew     cabin_crew         156
gate_agent     passenger           94
pilot          passenger           52
passenger      pilot               31
airport_staff  cabin_crew          19
cabin_crew     airport_staff       14
               pilot                8
airport_staff  airport_staff        5
pilot          cabin_crew           2
dtype: int64

In [30]:
progress = load_progress()

underfilled = []

for key, value in progress.items():
    if value["current_count"] < value["target_count"]:
        underfilled.append(
            {
                "key": key,
                "current_count": value["current_count"],
                "target_count": value["target_count"],
                "missing": value["target_count"] - value["current_count"],
                "status": value["status"],
            }
        )

underfilled_df = pd.DataFrame(underfilled)

if len(underfilled_df):
    display(underfilled_df.sort_values("missing", ascending=False))
else:
    print("No underfilled chunks.")

,key,current_count,target_count,missing,status
0,pets_and_service_animals::pet is anxious::Engl...,44,291,247,running


In [31]:
import shutil
from google.colab import files

ZIP_PATH = "/content/airplane_translation_dataset.zip"

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=str(EXPORT_DIR),
)

files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>